# 09 - Cross-city EDA: Chicago vs New York City vs Los Angeles

**Owner:** Bella - **Date:** 2026-07-18 - **Workstream:** Modeling (cross-city)

The platform serves three cities from three separate inspection feeds. They do
**not** share a grading system or a raw feature encoding, so the same modeling
recipe lands differently in each. This notebook compares them on the axes that
matter for the model, with presentation-ready charts saved to
`reports/figures/cross_city/`.

**What "a bad outcome" means in each city** (the thing the model predicts):
- **Chicago** - the next inspection is a **Fail** or cites a **priority violation** (codes 1-29), within 180 days.
- **New York City** - the next inspection is graded **B or C** (score >= 14).
- **Los Angeles** - the next inspection **scores below 90** out of 100.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
import build_nyc_scores as nyc
import build_la_scores as la
from foodsafety.config import FEATURES_PATH, PROCESSED_DIR, RAW_DIR

PROCESSED = Path(PROCESSED_DIR)
RAW = Path(RAW_DIR)
FIG = ROOT / "reports" / "figures" / "cross_city"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11.5,
                     "axes.titlesize": 13, "axes.titleweight": "bold"})
COL = {"Chicago": "#2166AC", "New York City": "#4D9221", "Los Angeles": "#B2182B"}
# what the label counts as a "bad outcome", for chart subtitles / axis labels
MECH = {"Chicago": "Fail / priority code",
        "New York City": "Next grade B or C",
        "Los Angeles": "Next score under 90"}
print("setup ok")

## Build one comparable frame per city
Each city keeps its own label + feature names; a few axes are standardized (a current-bad flag, prior bad-rate, inspection month) so the charts line up.

In [ ]:
chi = pd.read_parquet(FEATURES_PATH)
if "right_truncated" in chi.columns:
    chi = chi[~chi["right_truncated"]].copy()
chi["month"] = pd.to_datetime(chi["inspection_date"]).dt.month
chi["prior_rate"] = chi["prior_fails"] / chi["prior_inspections"].replace(0, np.nan)
chi["serious_now"] = (chi["n_priority_this_inspection"] > 0).astype(int)

nev = nyc.build_events()[0]
nev = nev[nev["next_score"].notna()].copy()
nev["month"] = nev["inspection_date"].dt.month
nev["serious_now"] = (nev["cur_n_critical"] > 0).astype(int)

lraw = la.build_raw()
lev = la.build_events(lraw)[0]
lev = lev[lev["next_score"].notna()].copy()
lev["month"] = lev["inspection_date"].dt.month
lev["serious_now"] = (lev["cur_n_critical"] > 0).astype(int)

CITY = {
    "Chicago": dict(df=chi, label="y_fail_or_critical_next_180d", cur="was_fail", prior="prior_rate"),
    "New York City": dict(df=nev, label="y_next_bc", cur="cur_is_bad", prior="prior_bad_rate"),
    "Los Angeles": dict(df=lev, label="y_next_bad", cur="cur_is_bad", prior="prior_bad_rate"),
}
for c, d in CITY.items():
    print(f"{c:14} rows={len(d['df']):>7,}  bad-outcome rate={d['df'][d['label']].mean():.1%}")

### 1. How often a bad outcome follows an inspection
The share of inspections that are followed by a bad outcome (each city's own definition). This is the base rate: the single biggest reason raw scores aren't comparable across cities.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
rates = {c: CITY[c]["df"][CITY[c]["label"]].mean() * 100 for c in CITY}
bars = ax.bar(list(rates), list(rates.values()), color=[COL[c] for c in rates], width=0.55)
for b, c in zip(bars, rates):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.5,
            f"{rates[c]:.0f}%", ha="center", va="bottom", fontweight="bold", fontsize=12)
ax.set_xticks(range(len(rates)))
ax.set_xticklabels([f"{c}\n{MECH[c]}" for c in rates], fontsize=9.5)
ax.set_ylabel("Inspections followed by a bad outcome (%)")
ax.set_title("How often a bad outcome follows an inspection")
ax.margins(y=0.18)
plt.tight_layout(); plt.savefig(FIG / "01_base_rate.png", bbox_inches="tight"); plt.show()

### 2. What raises the risk of a bad next inspection?
Two levers, side by side: the **current** inspection's own result, and the establishment's **prior** track record. Each bar is the bad-outcome rate for that group. Chicago is driven most by the current inspection; NYC and LA lean more on the prior record (they inspect roughly once a year, so 'now' is staler).

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.3))
groups = ["Clean inspection now", "Bad inspection now", "Poor prior record"]
gcolors = ["#BBBBBB", "#E08214", "#8073AC"]
x = np.arange(len(CITY)); w = 0.26
for gi, g in enumerate(groups):
    vals = []
    for c, d in CITY.items():
        df = d["df"]
        if g == "Clean inspection now":
            v = df.loc[df[d["cur"]].fillna(0) == 0, d["label"]].mean()
        elif g == "Bad inspection now":
            v = df.loc[df[d["cur"]].fillna(0) == 1, d["label"]].mean()
        else:  # poor prior record = top quartile of prior bad-rate
            thr = df[d["prior"]].quantile(0.75)
            v = df.loc[df[d["prior"]] >= max(thr, 0.001), d["label"]].mean()
        vals.append(v * 100)
    bars = ax.bar(x + (gi - 1) * w, vals, w, label=g, color=gcolors[gi])
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.6, f"{v:.0f}", ha="center", fontsize=8.5)
ax.set_xticks(x); ax.set_xticklabels(list(CITY))
ax.set_ylabel("Bad-outcome rate for the group (%)")
ax.set_title("What raises the risk of a bad next inspection?")
ax.legend(frameon=False, fontsize=10, loc="upper left")
ax.margins(y=0.15)
plt.tight_layout(); plt.savefig(FIG / "02_risk_drivers.png", bbox_inches="tight"); plt.show()

### 3. A New York City shutdown means *lower* risk next time
When NYC's health department closes an establishment, it must fix the problems and pass a re-inspection to reopen, so its next graded inspection is usually clean. That makes 'closed at this inspection' a strong, clean predictor (it splits future risk almost 2:1). Chicago and LA have no equivalent shutdown field in their feeds.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
d = CITY["New York City"]["df"]
g = d.groupby(d["cur_closed"].astype(int))["y_next_bc"].mean() * 100
share = d["cur_closed"].mean() * 100
bars = ax.bar(["Not closed", "Closed by the health dept."],
              [g.get(0, np.nan), g.get(1, np.nan)], color=["#C6DBB5", "#4D9221"], width=0.6)
for b, v in zip(bars, [g.get(0, 0), g.get(1, 0)]):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.7, f"{v:.0f}%",
            ha="center", va="bottom", fontweight="bold", fontsize=12)
ax.set_ylabel("Next inspection graded B or C (%)")
ax.set_title("New York City: a shutdown means lower risk next time")
ax.text(0.5, -0.22, f"Only {share:.0f}% of inspections end in a shutdown, but it is a clean signal.",
        transform=ax.transAxes, ha="center", fontsize=9.5, color="#555")
ax.margins(y=0.18)
plt.tight_layout(); plt.savefig(FIG / "03_nyc_closure.png", bbox_inches="tight"); plt.show()

### 4. When in the year is risk highest?
Bad-outcome rate by inspection month, with each city scaled to its **own yearly average** so the shapes are comparable (1.0 = that city's average month). All three wobble month to month, but only **Chicago's pattern is stable enough across years to help the model** - it passed the cross-validation gate. LA's swing looks the biggest here, but it is mostly small-sample noise (at a 9% base rate, monthly estimates are jumpy), so calendar features failed the gate for both NYC and LA.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.3))
mo = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
for c, d in CITY.items():
    df = d["df"]
    g = df.groupby("month")[d["label"]].mean()
    g = g / g.mean()
    ax.plot(g.index, g.values, "-o", color=COL[c], label=c, lw=2.2, ms=5)
ax.axhline(1.0, color="#999", ls=":", lw=1)
ax.set_xticks(range(1, 13)); ax.set_xticklabels(mo)
ax.set_xlabel("Month of inspection")
ax.set_ylabel("Bad-outcome rate vs the city's yearly average\n(1.0 = an average month)")
ax.set_title("When in the year is a bad outcome most likely?")
ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(FIG / "04_seasonality.png", bbox_inches="tight"); plt.show()

### 5. How well the served model ranks risk, per city
Two ranking-quality scores on each city's held-out test set. **ROC-AUC** is base-rate-independent (fair to compare across cities); **PR-AUC** rewards finding the minority bad cases and so tracks each city's prevalence.

In [ ]:
import json
perf = {}
for c, path in [("Chicago", "app/public/data/methodology.json"),
                ("New York City", "app/public/data/nyc/methodology.json"),
                ("Los Angeles", "app/public/data/la/methodology.json")]:
    h = json.loads((ROOT / path).read_text()).get("headline", {})
    perf[c] = {"PR-AUC": h.get("pr_auc"), "ROC-AUC": h.get("roc_auc")}
pdf = pd.DataFrame(perf).T
fig, ax = plt.subplots(figsize=(8, 4.3))
x = np.arange(len(pdf)); w = 0.36
ax.bar(x - w / 2, pdf["PR-AUC"], w, label="PR-AUC (finds the minority bad cases)", color="#08519C")
ax.bar(x + w / 2, pdf["ROC-AUC"], w, label="ROC-AUC (base-rate-independent ranking)", color="#9ECAE1")
for i, c in enumerate(pdf.index):
    ax.text(i - w / 2, pdf["PR-AUC"][c] + 0.015, f"{pdf['PR-AUC'][c]:.2f}", ha="center", fontsize=9.5)
    ax.text(i + w / 2, pdf["ROC-AUC"][c] + 0.015, f"{pdf['ROC-AUC'][c]:.2f}", ha="center", fontsize=9.5)
ax.set_xticks(x); ax.set_xticklabels(pdf.index)
ax.set_ylim(0, 1); ax.set_ylabel("Score (higher is better)")
ax.set_title("How well the served model ranks risk, per city")
ax.legend(frameon=False, fontsize=9.5, loc="upper right")
plt.tight_layout(); plt.savefig(FIG / "05_model_performance.png", bbox_inches="tight"); plt.show()
pdf.round(3)

**Why the two bars differ, and why cities differ.**

- **Within a city, PR-AUC vs ROC-AUC differ because of the base rate.** ROC-AUC
  asks "does the model rank a random bad establishment above a random good one?"
  and ignores how common bad ones are. PR-AUC asks "of the ones the model flags,
  how many are actually bad?" which gets harder as bad outcomes get rarer. So
  where bad outcomes are rare (**Chicago 13%, LA 9%**) PR-AUC sits far below
  ROC-AUC; where they are common (**NYC ~41% in this window**) the two are close.
- **Across cities, ROC-AUC is the fair comparison.** Chicago ranks best
  (ROC ~0.81) because a venue's current inspection strongly predicts the next;
  NYC (~0.70) and LA (~0.69) inspect less often, so their signal is noisier. LA's
  low PR-AUC is mostly its low base rate, not a weaker model - its ROC-AUC is
  right next to NYC's.

### 6. Serious violations are more common where grading is stricter
Share of inspections that cite at least one **serious** violation (Chicago: a priority code 1-29; NYC / LA: a critical / major violation). A rough cross-city read on how strict each program is at the point of inspection.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sr = {c: CITY[c]["df"]["serious_now"].mean() * 100 for c in CITY}
bars = ax.bar(list(sr), list(sr.values()), color=[COL[c] for c in sr], width=0.6)
for b, v in zip(bars, sr.values()):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.8, f"{v:.0f}%",
            ha="center", va="bottom", fontweight="bold", fontsize=12)
ax.set_xticks(range(len(sr))); ax.set_xticklabels(list(sr))
ax.set_ylabel("Inspections citing a serious violation (%)")
ax.set_title("How often inspections cite a serious violation")
ax.margins(y=0.18)
plt.tight_layout(); plt.savefig(FIG / "06_serious_violation_rate.png", bbox_inches="tight"); plt.show()

### 7. Violation category mix across cities
All three cities' native violation codes map to one shared set of categories via `reference/violation_crosswalk.csv`, so their mix is comparable. Share of inspections citing each category (Chicago codes are parsed from the free-text `violations` field; NYC/LA from their `violation_code`). Big gaps (e.g. NYC rarely coding 'equipment / nonfood surface') reflect each city's coding practice as much as real conditions.

In [ ]:
import re
cw = pd.read_csv(ROOT / "reference/violation_crosswalk.csv")
cw["native_code"] = cw["native_code"].astype(str)
tmap = {c: cw[cw.city == c].set_index("native_code")["theme"].to_dict() for c in ["Chicago", "NYC", "LA"]}

def share_from_codes(insp_codes, theme_map):
    n = len(insp_codes); counts = {}
    for codes in insp_codes:
        for th in {theme_map.get(str(c)) for c in codes} - {None}:
            counts[th] = counts.get(th, 0) + 1
    return {th: v / n * 100 for th, v in counts.items()}

chi_v = pd.read_parquet(PROCESSED / "inspections_labeled.parquet", columns=["violations"])["violations"].dropna()
pat = re.compile(r"(?:^|\|)\s*(\d{1,2})\.\s")
CHI = share_from_codes(chi_v.map(lambda t: pat.findall(t)), tmap["Chicago"])

nyc = pd.read_parquet(RAW / "nyc_inspections.parquet", columns=["camis", "inspection_date", "violation_code"]).dropna(subset=["violation_code"])
NYC = share_from_codes(nyc.astype({"violation_code": str}).groupby(["camis", "inspection_date"])["violation_code"].apply(list), tmap["NYC"])

li = pd.read_parquet(RAW / "la_inspections.parquet", columns=["serial_number"])
lv = pd.read_parquet(RAW / "la_violations.parquet", columns=["serial_number", "violation_code"]).astype({"violation_code": str})
la_codes = lv.groupby("serial_number")["violation_code"].apply(list).reindex(li["serial_number"].unique()).map(lambda x: x if isinstance(x, list) else [])
LA = share_from_codes(la_codes, tmap["LA"])

DATA = {"Chicago": CHI, "New York City": NYC, "Los Angeles": LA}
themes = sorted({t for d in DATA.values() for t in d}, key=lambda t: -sum(d.get(t, 0) for d in DATA.values()))[:8]
CL = {"Chicago": "#8AB2DA", "New York City": "#A9D18C", "Los Angeles": "#E4A0A5"}  # softer tones
fig, ax = plt.subplots(figsize=(9.8, 5.6))
y = np.arange(len(themes)); h = 0.24
for ci, (c, d) in enumerate(DATA.items()):
    ax.barh(y + (1 - ci) * h, [d.get(t, 0) for t in themes], h, label=c,
            color=CL[c], edgecolor="white", linewidth=0.6)
ax.set_yticks(y); ax.set_yticklabels([t.replace("_", " ").title() for t in themes])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Inspections citing this category (%)")
ax.set_title("Violation category mix across cities", pad=10)
ax.grid(axis="x", color="#E9E9E9", linewidth=0.9)
ax.set_axisbelow(True)
ax.tick_params(length=0)
ax.spines["left"].set_color("#CCCCCC")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.savefig(FIG / "07_violation_categories.png", bbox_inches="tight"); plt.show()

## Model results: how the served XGBoost performs in each city

Everything below is on each city's **held-out test set**, read from the committed
`methodology.json` (ranking + operating-point metrics). These are the numbers the
promotion **gate** and the methodology page use.
The gate (decision 0002) requires a candidate to clear the incumbent on **both**
PR-AUC and precision@10%.

In [ ]:
import glob
def load_metrics():
    meth = {"Chicago": "app/public/data/methodology.json",
            "New York City": "app/public/data/nyc/methodology.json",
            "Los Angeles": "app/public/data/la/methodology.json"}
    run = {"Chicago": sorted(glob.glob(str(ROOT / "reports/metrics/xgb/xgb_monotone_sigmoid_20260718_*.json")))[-1],
           "New York City": str(ROOT / "reports/metrics/nyc/nyc_20260718_a8e91fbc2.json")}
    M = {}
    for c, p in meth.items():
        d = json.loads((ROOT / p).read_text())
        ops = {round(o["frac"], 2): o for o in d["operating_points"]}
        M[c] = dict(pr_auc=d["headline"]["pr_auc"], roc_auc=d["headline"]["roc_auc"],
                    lift10=d["headline"]["top_decile_lift"], prevalence=d["test"]["prevalence"],
                    n=d["test"]["n"], events=d["test"]["events"], ops=ops, brier=None)
        if c in run:
            M[c]["brier"] = json.loads(Path(run[c]).read_text())["test"].get("brier_score")
    return M
MET = load_metrics()
for c, r in MET.items():
    print(f"{c:14} PR-AUC={r['pr_auc']:.3f}  ROC-AUC={r['roc_auc']:.3f}  base={r['prevalence']:.1%}")

### 8. Model scorecard across cities
The headline ranking + operating-point metrics side by side, all at the top-10% operating point where noted. **PR-AUC and Precision@10% are the promotion gate; the rest are reporting metrics.** F1@10% balances precision and recall at that operating point. Lift is on a 1x+ scale so it sits in its own panel (1x = no better than random). ROC-AUC is the fair cross-city comparison (base-rate-independent); PR-AUC and precision track each city's prevalence.

In [ ]:
def f1_at(r, frac):
    p, rc = r["ops"][frac]["precision"], r["ops"][frac]["recall"]
    return 2 * p * rc / (p + rc) if (p + rc) else 0.0
fig, (ax, axL) = plt.subplots(1, 2, figsize=(12, 4.7), gridspec_kw={"width_ratios": [3.3, 1]})
groups = ["PR-AUC\n(gate)", "ROC-AUC", "Precision@10%\n(gate)", "Recall@10%", "F1 @ 10%"]
x = np.arange(len(groups)); w = 0.26
for ci, (c, r) in enumerate(MET.items()):
    vals = [r["pr_auc"], r["roc_auc"], r["ops"][0.1]["precision"], r["ops"][0.1]["recall"], f1_at(r, 0.1)]
    bars = ax.bar(x + (ci - 1) * w, vals, w, label=c, color=COL[c])
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.012, f"{v:.2f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(groups, fontsize=9.5)
ax.set_ylim(0, 1); ax.set_ylabel("Score (higher is better)")
ax.set_title("Served-model scorecard across cities")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.0))
# lift is on a 1x+ scale, so it gets its own small panel
_short = {"Chicago": "Chicago", "New York City": "NYC", "Los Angeles": "LA"}
xl = np.arange(len(MET))
for ci, (c, r) in enumerate(MET.items()):
    axL.bar(ci, r["ops"][0.1]["lift"], 0.62, color=COL[c])
    axL.text(ci, r["ops"][0.1]["lift"] + 0.05, f"{r['ops'][0.1]['lift']:.1f}x", ha="center", fontsize=9)
axL.axhline(1.0, color="#999", ls=":", lw=1.2)
axL.text(len(MET) - 0.5, 1.06, "1x = random", color="#777", fontsize=8, ha="right")
axL.set_xticks(xl); axL.set_xticklabels([_short[c] for c in MET], fontsize=9)
axL.set_ylim(0, max(r["ops"][0.1]["lift"] for r in MET.values()) * 1.25)
axL.set_title("Lift @ 10%")
plt.tight_layout(); plt.savefig(FIG / "08_scorecard.png", bbox_inches="tight"); plt.show()

### 9. Top-K lift: how much better than random, per operating point
Working the top K% of establishments by predicted risk, the **lift** is how many more bad outcomes you catch than picking at random (1.0 = no better than random). Steeper on the left = the model concentrates real risk at the very top of the list.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
for c, r in MET.items():
    fr = sorted(r["ops"]); lifts = [r["ops"][f]["lift"] for f in fr]
    ax.plot([f * 100 for f in fr], lifts, "-o", color=COL[c], label=c, lw=2.2, ms=5)
ax.axhline(1.0, color="#999", ls=":", lw=1.2)
ax.text(48, 1.05, "random (no lift)", color="#777", fontsize=9, ha="right")
ax.set_xlabel("Top K% of establishments worked (by predicted risk)")
ax.set_ylabel("Lift over random\n(x more bad outcomes caught)")
ax.set_title("Top-K lift by city")
ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(FIG / "09_topk_lift.png", bbox_inches="tight"); plt.show()

### 10. Confusion matrix at the top-10% operating point
If the city worked the **top 10%** of establishments by predicted risk, this is what the next-inspection outcomes would look like. Derived exactly from the operating-point counts (flagged, caught) - no re-fitting. Precision = TP / (TP+FP); recall = TP / (TP+FN).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, (c, r) in zip(axes, MET.items()):
    op = r["ops"][0.1]
    N, pos = r["n"], r["events"]
    flagged, tp = op["n_flagged"], op["events_caught"]
    fp = flagged - tp; fn = pos - tp; tn = N - flagged - fn
    # rows = predicted (flagged / not), cols = actual (bad / ok)
    cm = np.array([[tp, fp], [fn, tn]])
    ax.imshow(cm, cmap="Blues", aspect="auto")
    for (i, j), lab in [((0, 0), "TP"), ((0, 1), "FP"), ((1, 0), "FN"), ((1, 1), "TN")]:
        ax.text(j, i, f"{lab}\n{cm[i, j]:,}", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() * 0.5 else "#222", fontsize=11, fontweight="bold")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Actually\nbad", "Actually\nok"], fontsize=9)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Flagged\n(top 10%)", "Not\nflagged"], fontsize=9)
    ax.set_title(f"{c}\nprecision {op['precision']:.0%}  recall {op['recall']:.0%}", fontsize=11)
fig.suptitle("Confusion matrix at the top-10% operating point", fontweight="bold", y=1.06)
plt.tight_layout(); plt.savefig(FIG / "10_confusion_top10.png", bbox_inches="tight"); plt.show()

### 11. All gating + reporting metrics, per city
Every metric the gate and the methodology page track, in one table. **Gold** = the promotion gate (PR-AUC + precision@10%); **darker blue** = the two headline metrics at the top of the how-it-works scorecard (ROC-AUC + top-decile lift); **lighter blue** = the other reported metrics shown elsewhere in the app. Unhighlighted rows are context (test-set size, base rate). The note under the chart explains why the gate and headline metrics were chosen.

In [ ]:
rows = [("Test establishments", lambda r: f"{r['n']:,}"),
        ("Bad-outcome base rate", lambda r: f"{r['prevalence']:.1%}"),
        ("PR-AUC  (gate)", lambda r: f"{r['pr_auc']:.3f}"),
        ("ROC-AUC", lambda r: f"{r['roc_auc']:.3f}"),
        ("Precision @ top 5%", lambda r: f"{r['ops'][0.05]['precision']:.2f}"),
        ("Precision @ top 10%  (gate)", lambda r: f"{r['ops'][0.1]['precision']:.2f}"),
        ("Precision @ top 20%", lambda r: f"{r['ops'][0.2]['precision']:.2f}"),
        ("Recall @ top 5%", lambda r: f"{r['ops'][0.05]['recall']:.2f}"),
        ("Recall @ top 10%", lambda r: f"{r['ops'][0.1]['recall']:.2f}"),
        ("Recall @ top 20%", lambda r: f"{r['ops'][0.2]['recall']:.2f}"),
        ("Lift @ top 5%", lambda r: f"{r['ops'][0.05]['lift']:.2f}x"),
        ("Lift @ top 10%", lambda r: f"{r['ops'][0.1]['lift']:.2f}x"),
        ("Lift @ top 20%", lambda r: f"{r['ops'][0.2]['lift']:.2f}x")]
cities = list(MET)
cell = [[fn(MET[c]) for c in cities] for _, fn in rows]
from matplotlib.patches import Patch
GATE_C, HERO_C, REPORT_C = "#F6C650", "#BBD9F0", "#E4F0FA"
_context = {"Test establishments", "Bad-outcome base rate"}
HERO = {"ROC-AUC", "Lift @ top 10%"}   # the two headline numbers in the how-it-works scorecard
fig, ax = plt.subplots(figsize=(9.2, 6.2)); ax.axis("off")
tbl = ax.table(cellText=cell, rowLabels=[r[0] for r in rows], colLabels=cities,
               cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 1.4)
for j, c in enumerate(cities):
    tbl[(0, j)].set_facecolor(COL[c]); tbl[(0, j)].set_text_props(color="white", fontweight="bold")
for i, (name, _) in enumerate(rows):
    is_gate = "gate" in name
    color = GATE_C if is_gate else (HERO_C if name in HERO else (None if name in _context else REPORT_C))
    if color:
        tbl[(i + 1, -1)].set_facecolor(color)
        for j in range(len(cities)):
            tbl[(i + 1, j)].set_facecolor(color)
    if is_gate:
        tbl[(i + 1, -1)].set_text_props(fontweight="bold")
ax.legend(handles=[Patch(facecolor=GATE_C, edgecolor="#999", label="Gating metric (promotion gate)"),
                   Patch(facecolor=HERO_C, edgecolor="#999", label="Headline metric (top of the how-it-works scorecard)"),
                   Patch(facecolor=REPORT_C, edgecolor="#999", label="Other reported metric (elsewhere in the app)")],
          loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=1, frameon=False, fontsize=9.5, handlelength=1.5)
note = ("Gate (decision 0002): a model ships only if it beats the incumbent on BOTH gate metrics — "
        "PR-AUC (overall ranking of the rare risky cases) and precision@10% (how much of the top-10% worklist is truly risky).\n"
        "The two headline metrics lead the methodology page: ROC-AUC because it is base-rate-independent (the fair cross-city "
        "comparison), and top-decile lift because it is the most intuitive ‘x better than random.’")
ax.text(0.5, -0.30, note, transform=ax.transAxes, ha="center", va="top", fontsize=9, color="#333")
ax.set_title("All gating + reporting metrics (held-out test)", fontweight="bold", pad=14)
plt.savefig(FIG / "11_metrics_table.png", bbox_inches="tight", dpi=130); plt.show()

### 12. Most impactful experiments: what shipped vs what didn't help
**Green** = kept / shipped; **grey** = tried but reverted. The upper table lists each city's served models on its own held-out test (base rates differ, so PR-AUC / precision are not comparable between cities — ROC-AUC is). The lower table shows each experiment's Δ PR-AUC vs its own control.

In [ ]:
from matplotlib.patches import Patch
def _T(pat):
    fs = sorted(glob.glob(str(ROOT / pat)))
    return json.loads(Path(fs[-1]).read_text())["test"]
def _f1(t):
    p, r = t["precision_at_10pct"], t["recall_at_10pct"]
    return 2 * p * r / (p + r) if (p + r) else 0.0
def _mrow(t):
    return [f"{t['pr_auc']:.3f}", f"{t['roc_auc']:.3f}", f"{t['precision_at_10pct']:.3f}",
            f"{t['recall_at_10pct']:.3f}", f"{_f1(t):.3f}", f"{t['top_decile_lift']:.2f}x"]
GREEN, GRAY, HEADG, HEADR = "#E7F2E1", "#EFEFEF", "#4D9221", "#8A8A8A"
mcols = ["Base rate", "PR-AUC", "ROC-AUC", "Prec@10", "Rec@10", "F1@10", "Lift@10"]
shipped = [
    ("Chicago", "LogReg baseline", "reports/metrics/chicago/chicago_logreg_full_*.json", False),
    ("Chicago", "XGBoost + colsample 0.70  (HPO, served)", "reports/metrics/xgb/xgb_monotone_sigmoid_20260718_*.json", True),
    ("NYC", "XGBoost, before features", "reports/metrics/nyc/nyc_served_xgb_full_20260712_*.json", False),
    ("NYC", "+ closure/tenure + depth-2  (served)", "reports/metrics/nyc/nyc_20260718_a8e91fbc2.json", True),
    ("LA", "XGBoost  (served)", "reports/metrics/la/la_served_xgb_full_2026071*.json", True),
]
A_rows, A_cell, A_bold = [], [], []
for city, name, pat, served in shipped:
    t = _T(pat); A_rows.append(f"{city}: {name}")
    A_cell.append([f"{t['positive_rate']*100:.1f}%"] + _mrow(t)); A_bold.append(served)
didnt = [
    ("Block-face building permits + violations", "Chicago", "+0.002", "physical-plant condition"),
    ("Violation free-text embeddings (Titan)", "Chicago", "+0.000", "dense NLP on comments"),
    ("LLM violation severity labels (Nova)", "Chicago", "+0.002", "structured hazard/severity"),
    ("Exposure / inverse-propensity reweighting", "Chicago", "-0.011", "de-confound inspection arrival"),
    ("Calendar / seasonality features", "NYC & LA", "-0.001", "month + quarter"),
    ("Recent-window (365-day) priors", "NYC & LA", "-0.003", "recency vs lifetime history"),
]
fig = plt.figure(figsize=(12.5, 7.6))
gs = fig.add_gridspec(2, 1, height_ratios=[5.5, 6], hspace=0.55)
axA = fig.add_subplot(gs[0]); axA.axis("off")
tA = axA.table(cellText=A_cell, rowLabels=A_rows, colLabels=mcols, cellLoc="center", loc="center")
tA.auto_set_font_size(False); tA.set_fontsize(9.5); tA.scale(1, 1.5)
for j in range(len(mcols)):
    tA[(0, j)].set_facecolor(HEADG); tA[(0, j)].set_text_props(color="white", fontweight="bold")
for i in range(len(A_rows)):
    for j in list(range(len(mcols))) + [-1]:
        tA[(i + 1, j)].set_facecolor(GREEN)
        if A_bold[i]:
            tA[(i + 1, j)].set_text_props(fontweight="bold")
axA.set_title("What shipped — served models (each city's own held-out test; bold = current served)", fontweight="bold", fontsize=12, pad=8)
axB = fig.add_subplot(gs[1]); axB.axis("off")
Bcols = ["City", "What it added", "Δ PR-AUC", "Verdict"]
Bcell = [[c, added, dpr, "reverted"] for name, c, dpr, added in didnt]
tB = axB.table(cellText=Bcell, rowLabels=[d[0] for d in didnt], colLabels=Bcols, cellLoc="center", loc="center")
tB.auto_set_font_size(False); tB.set_fontsize(9.5); tB.scale(1, 1.5)
for j in range(len(Bcols)):
    tB[(0, j)].set_facecolor(HEADR); tB[(0, j)].set_text_props(color="white", fontweight="bold")
for i in range(len(didnt)):
    for j in list(range(len(Bcols))) + [-1]:
        tB[(i + 1, j)].set_facecolor(GRAY)
axB.set_title("Tried but didn't help — reverted (Δ PR-AUC vs each experiment's own control)", fontweight="bold", fontsize=12, pad=8)
fig.legend(handles=[Patch(facecolor=GREEN, edgecolor="#999", label="Kept / shipped"),
                    Patch(facecolor=GRAY, edgecolor="#999", label="Tried, didn't help (reverted)")],
           loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False, fontsize=10)
fig.suptitle("Most impactful feature + hyperparameter experiments", fontweight="bold", fontsize=14, y=0.99)
plt.savefig(FIG / "12_experiments_table.png", bbox_inches="tight", dpi=130); plt.show()

### 13. Which features matter most (leave-one-out on the risk model)
From `scripts/run_leave_one_out.py`: refit each city's served risk model dropping one feature at a time, and measure the PR-AUC it loses. The biggest bar = the most important feature. Chicago hinges on the current inspection's own outcome; NYC's single most important feature is the DOHMH closure flag (the 2026-07-18 addition); LA's signal is spread across recency + prior history. Caveat: leave-one-out understates *correlated* features (single-split diagnostic).

In [ ]:
loo = json.loads((ROOT / "reports/metrics/leave_one_out_risk.json").read_text())
LABELS = {"was_fail": "Failed this inspection", "n_priority_this_inspection": "Priority violations now",
    "n_core_this_inspection": "Core violations now", "static_inspection_type": "Inspection type",
    "flag_kw_cooling": "'Cooling' keyword flag", "license_age_days": "License age",
    "prev_priority_violations": "Prev priority violations", "days_since_last_fail": "Days since last fail",
    "cur_closed": "Closed by health dept.", "prior_closures": "Prior closures", "tenure_days": "Establishment tenure",
    "prior_inspections": "Prior inspections", "prior_bad_rate": "Prior bad-rate", "prior_bad": "Prior bad inspections",
    "prior_n_critical": "Prior critical violations", "prior_mean_score": "Prior mean score",
    "prev_score": "Previous score", "prev_is_bad": "Previous inspection was bad", "cur_score": "Current score",
    "days_since_last_inspection": "Days since last inspection"}
def pretty(f):
    if f in LABELS: return LABELS[f]
    if f.startswith("cur_theme_"): return f.replace("cur_theme_", "").replace("_", " ").capitalize() + " (now)"
    if f.startswith("prior_cur_sev_"): return "Prior " + f.split("_")[-1] + "-tier violations"
    if f.startswith("cur_sev_"): return f.split("_")[-1] + "-tier violations (now)"
    return f.replace("_", " ").capitalize()
cities = [("Chicago", "chicago"), ("New York City", "nyc"), ("Los Angeles", "la")]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.7))
for ax, (disp, key) in zip(axes, cities):
    top = sorted(loo[key]["deltas"].items(), key=lambda kv: -kv[1])[:6][::-1]
    names = [pretty(f) for f, _ in top]; vals = [v for _, v in top]
    ax.barh(range(len(names)), vals, color=COL[disp], height=0.72)
    for i, v in enumerate(vals):
        ax.text(v + max(vals) * 0.02, i, f"{v:.3f}", va="center", fontsize=8.5)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8.7)
    ax.set_xlim(0, max(vals) * 1.28)
    ax.set_title(f"{disp}  (full model {loo[key]['baseline_pr_auc']:.3f})", fontsize=11.5)
    ax.grid(axis="x", color="#ECECEC", lw=0.9); ax.set_axisbelow(True); ax.tick_params(length=0)
    if key == "la":
        ax.tick_params(axis="x", labelsize=7)
    if ax is axes[0]:
        ax.set_xlabel("PR-AUC lost when the feature is removed")
fig.suptitle("Which features matter most — leave-one-out on the risk model", fontweight="bold", y=1.02)
plt.tight_layout(); plt.savefig(FIG / "13_ablation.png", bbox_inches="tight", dpi=130); plt.show()

## Takeaways

1. **Bad-outcome rates span ~4x** across cities, so raw PR-AUC is not comparable - use ROC-AUC and lift.
2. **The dominant signal flips:** Chicago's current inspection carries it; NYC/LA lean on the prior track record.
3. **City-native signals matter:** NYC's shutdown flag is a strong, clean predictor (a shutdown means *lower* next-visit risk); Chicago and LA have no equivalent field. This is the 2026-07-18 NYC feature win.
4. **Only Chicago has a stable seasonal pattern** - all three vary by month, but NYC's and LA's swings are mostly small-sample noise, so calendar features passed the cross-validation gate only in Chicago.
5. **XGBoost + Platt is the right served recipe everywhere,** but the *features* should be picked per city from the same concept menu, not standardized to a lowest common denominator.